# H-Node Hallucination Detection — Colab Pipeline

End-to-end run of the H-Node hallucination detector built on vLLM-Hook. Three stages:

1. **extract** — capture last-token hidden states at every transformer layer for 600 labeled TruthfulQA prompts (~5–10 min on T4)
2. **train** — fit per-layer logistic-regression probes, pick the best layer, identify top-50 H-Nodes (~30 sec, CPU)
3. **detect** — load only the best layer and score new prompts (~1 min on T4)

**Important:** we pin `vllm==0.7.3` because Colab's default vLLM (v0.21+) enables async scheduling, which the hook framework's `execute_model` override doesn't yet support.

After installs, **Runtime → Restart Runtime** (mandatory) before running the rest.

## 1. GPU sanity check

In [1]:
!nvidia-smi

Tue May 19 04:52:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone the fork

In [2]:
%cd /content
!rm -rf vLLM-Hook
!git clone https://github.com/Samarpit-bhatia/vLLM-Hook.git
%cd /content/vLLM-Hook
!git log --oneline -5

/content
Cloning into 'vLLM-Hook'...
remote: Enumerating objects: 315, done.
remote: Counting objects: 100% (180/180), done.
remote: Compressing objects: 100% (110/110), done.
remote: Total 315 (delta 95), reused 112 (delta 69), pack-reused 135 (from 2)
Receiving objects: 100% (315/315), 2.17 MiB | 23.15 MiB/s, done.
Resolving deltas: 100% (151/151), done.
/content/vLLM-Hook
ed09399 (HEAD -> main, origin/main, origin/HEAD) fix(halludetect): switch to TRITON_ATTN backend (V1-compatible on T4)
28a2349 fix(halludetect): use XFORMERS attention backend for T4 / SM<8.0 GPUs
596b52f fix(halludetect): pin transformers==4.49.0 alongside vllm==0.7.3
ac29f8c fix(halludetect): resolve vllm_hook_plugins package shadowing
5ccb320 fix(halludetect): disable async scheduling for v0.21+ compatibility


## 3. Install dependencies (with vLLM pin)

After this cell finishes, **restart the runtime** (Runtime → Restart runtime) before running anything else. Python has cached the wrong vllm and won't pick up the new one without a restart.

In [3]:
# Pin to a vLLM the hook framework was tested against (pre-async-scheduling)
# AND pin transformers to a compatible version. T4 (SM 7.5) uses the
# TRITON_ATTN backend (set via env var in the demo script) since FlashAttention
# 2/3 need SM >= 8.0 and V1 doesn't accept XFORMERS.
!pip uninstall -y vllm transformers 2>/dev/null || true
!pip install "vllm==0.7.3" "transformers==4.49.0" --quiet
!pip install -e /content/vLLM-Hook/vllm_hook_plugins --quiet
!pip install datasets scikit-learn --quiet
print("\n>>> NOW RESTART THE RUNTIME (Runtime → Restart runtime), then continue from section 4.")

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.6/264.6 MB 5.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 123.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 11.3 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 4. Verify environment (after restart)

In [1]:
import torch, vllm, transformers
print("torch:       ", torch.__version__, "| cuda:", torch.cuda.is_available())
print("vllm:        ", vllm.__version__)
print("transformers:", transformers.__version__)
print("gpu:         ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(none)")
assert vllm.__version__.startswith("0.7"), f"Expected vllm 0.7.x, got {vllm.__version__} — restart runtime?"
assert transformers.__version__.startswith("4.49"), f"Expected transformers 4.49.x, got {transformers.__version__} — restart runtime?"

torch:        2.5.1+cu124 | cuda: True
vllm:         0.7.3
transformers: 4.49.0
gpu:          Tesla T4


## 5. Stage 1 — Extract (forward pass with hooks)

Loads Qwen2.5-3B, registers forward hooks on all 36 transformer layers via `ProbeHiddenStatesWorker`, runs ~600 TruthfulQA prompts, dumps last-token activations per layer to `hallucination_detection/artifacts/activations.pt`.

Total time on T4: ~10 min (mostly model load + warmup; the actual prompt loop is fast).

In [2]:
%cd /content/vLLM-Hook
!git pull
!rm -rf /dev/shm/vllm_hook
!python examples/demo_halludetect.py extract

/content/vLLM-Hook
Already up to date.
Building TruthfulQA labeled pairs...
README.md: 9.59kB [00:00, 27.4MB/s]
multiple_choice/validation-00000-of-0000(…): 100% 271k/271k [00:00<00:00, 327kB/s] 
Generating validation split: 100% 817/817 [00:00<00:00, 21906.22 examples/s]
  600 prompts (300 grounded / 300 hallucinated)
Loading model under ProbeHiddenStatesWorker (training config)...
INFO 05-19 04:59:58 __init__.py:207] Automatically detected platform cuda.
INFO 05-19 05:00:00 __init__.py:30] Available plugins for group vllm.general_plugins:
INFO 05-19 05:00:00 __init__.py:32] name=hook_registry, value=vllm_hook_plugins:register_plugins
INFO 05-19 05:00:00 __init__.py:34] all available plugins for group vllm.general_plugins will be loaded.
INFO 05-19 05:00:00 __init__.py:36] set environment variable VLLM_PLUGINS to control which plugins to load.
INFO 05-19 05:00:00 __init__.py:44] plugin hook_registry loaded.
WARNING 05-19 05:00:00 arg_utils.py:1385] Setting max_num_batched_tokens to 81

In [ ]:
# Confirm the extracted bundle exists and has the expected shape
import torch, os
p = "hallucination_detection/artifacts/activations.pt"
assert os.path.exists(p), f"Missing {p} — extract stage failed"
bundle = torch.load(p, map_location="cpu")
print("layers captured:", len(bundle["activations"]))
print("hidden size:    ", next(iter(bundle["activations"].values())).shape[1])
print("prompts:        ", len(bundle["labels"]))
print("grounded/hall:  ", int((bundle["labels"] == 0).sum()), "/", int((bundle["labels"] == 1).sum()))

## 6. Stage 2 — Train (probe fit, best-layer pick, H-Node ID)

Pure CPU (sklearn). Rewrites the inference config to capture only the chosen best layer.

In [ ]:
!python examples/demo_halludetect.py train

In [ ]:
# Inspect the trained probe
import json, numpy as np
with open("hallucination_detection/artifacts/probe.json") as f:
    meta = json.load(f)
print("best_layer:", meta["best_layer"])
print("AUC@best:  ", meta["auc_per_layer"][str(meta["best_layer"])])
print("n_h_nodes: ", meta["n_h_nodes"])

data = np.load("hallucination_detection/artifacts/probe.npz")
print("H-Nodes (first 10):", data["h_node_indices"][:10].tolist())

## 7. Stage 3 — Detect (live scoring)

Reloads the model with the *inference* config (only the best layer is hooked), then scores 6 example prompts: grounded answers should print P(hallucinated) near 0, wrong answers near 1.

In [ ]:
!rm -rf /dev/shm/vllm_hook   # fresh state
!python examples/demo_halludetect.py detect

## 8. Save artifacts back to your machine

In [ ]:
from google.colab import files
files.download("hallucination_detection/artifacts/probe.npz")
files.download("hallucination_detection/artifacts/probe.json")